# FXS / 11 — How big are the moves? Daily pip ranges of the 8 FXS 07 signals (2020 → 2026-11)

Iván (2026-11-12): the average range in pips, every day, for the 8 signals of FXS 07.

Per signal and day (1-minute bid/ask high/low; quoted Dukascopy spread, $3/lot):

| Measure | Meaning |
|---|---|
| **window range** | highest mid − lowest mid while the trade is open (pips) |
| **\|move\|** | \|mid at exit − mid at entry\| (pips), direction ignored |
| **MFE** | best open profit during the trade, at the exit quote (bid for a long, ask for a short) |
| **MAE** | worst open loss during the trade |
| **net** | what the time exit actually made, after spread and commission |
| **day range** | the pair's high − low over the whole New York day (17:00 → 17:00) |

The last columns show how much of the MFE survives to the exit, and how often the MFE reached 5, 10 and 20 pips. Descriptive only.

In [ ]:
import json, os, sys, tempfile
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

COMMISSION_PER_LOT = 3.0
WINDOWS = ((dt.date(2020, 1, 1), dt.date(2020, 12, 31), "discovery"), (dt.date(2021, 1, 1), dt.date(2025, 12, 31), "confirmation"),
           (dt.date(2026, 1, 1), dt.date(2026, 11, 6), "holdout"))
SIGNALS = {
    "EURUSD-02": ("EURUSD", "NY", 660, 900, -1), "USDJPY-01": ("USDJPY", "UTC", 55, 115, -1), "USDJPY-02": ("USDJPY", "UTC", 25, 55, 1),
    "N-EU-1": ("EURUSD", "NY", 885, 900, -1), "N-UJ-1": ("USDJPY", "UTC", 55, 70, -1), "N-UJ-3": ("USDJPY", "NY", 1340, 1400, 1),
    "N-EJ-1": ("EURJPY", "NY", 1315, 1435, 1), "N-EJ-2": ("EURJPY", "NY", 780, 900, -1)}

SYNTHETIC = os.environ.get("FXS_SYNTHETIC") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("FXS_SYNTH_DIR") or tempfile.mkdtemp(prefix="fxs11_")
    os.environ.pop("DUKASCOPY_DIR", None)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import fx_seasonality as F

cfg = F.FXSConfig()
OUT = Path.cwd() / ("11_synthetic" if SYNTHETIC else "11"); OUT.mkdir(exist_ok=True)
PAIRS = sorted({v[0] for v in SIGNALS.values()})
pd.set_option("display.width", 240, "display.max_columns", 40)

## 0. Synthetic inputs (smoke run only)

In [ ]:
if SYNTHETIC:
    base = Path(os.environ["EQUITY_TRADING_DIR"]) / "Dukascopy"
    for i, p in enumerate(PAIRS):
        F.write_synthetic_candles(base, p, dt.date(2019, 12, 28), dt.date(2026, 11, 6), seed=81 + i, price=150.0 if p.endswith("JPY") else 1.10)

## 1. Every trade, every day

In [ ]:
rows, day_rng = [], []
for a, b, st in WINDOWS:
    dates = pd.bdate_range(a, b)
    for p in PAIRS:
        m = F.load_minutes_ohlc(p, a, b, st, cfg.min_file_share)
        pip = F.pip_size(p)
        mh, ml = 0.5 * (m["bid_h"] + m["ask_h"]).to_numpy(), 0.5 * (m["bid_l"] + m["ask_l"]).to_numpy()
        mc = 0.5 * (m["bid_c"] + m["ask_c"]).to_numpy()
        idx = m.index
        # whole New York day range
        t0, t1 = F.entry_exit_times("NY", 0, 1440, dates)
        for d, x0, x1 in zip(dates, t0, t1):
            i0, i1 = idx.searchsorted(x0), idx.searchsorted(x1)
            if pd.isna(x0) or i1 - i0 < 600 or not m["tick"].to_numpy()[i0:i1].any():
                continue
            day_rng.append({"pair": p, "date": d, "day range": (np.nanmax(mh[i0:i1]) - np.nanmin(ml[i0:i1])) / pip})
        for sig, (pp, clock, s, e, direction) in SIGNALS.items():
            if pp != p:
                continue
            tr = F.run_structure(m, p, clock, s, e, direction, dates, F.NO_STOP, True, COMMISSION_PER_LOT)
            e0, e1 = F.entry_exit_times(clock, s, e, tr["date"])
            i0, i1 = idx.searchsorted(e0), idx.searchsorted(e1)
            wr = np.array([(max(mh[a_:b_].max(), mc[a_ - 1]) - min(ml[a_:b_].min(), mc[a_ - 1])) / pip for a_, b_ in zip(i0, i1)])
            mv = np.abs(mc[i1 - 1] - mc[i0 - 1]) / pip
            rows.append(tr.assign(signal=sig, pair=p, **{"window range": wr, "|move|": mv}))
T = pd.concat(rows, ignore_index=True)
T["year"] = pd.to_datetime(T["date"]).dt.year
DR = pd.DataFrame(day_rng); DR["year"] = pd.to_datetime(DR["date"]).dt.year
T = T[T["window range"] > 0]                     # days without quotes in the window
T.to_parquet(OUT / "11_trades.parquet")
print(T.groupby("signal").size())

## 2. Average day per signal (2020 → 2026)

In [ ]:
def summ(g):
    return pd.Series({"days": len(g), "window range mean": g["window range"].mean(), "window range median": g["window range"].median(),
                      "window range p90": g["window range"].quantile(0.9), "|move| mean": g["|move|"].mean(),
                      "MFE mean": g["best_pips"].mean(), "MFE median": g["best_pips"].median(), "MAE mean": g["worst_pips"].mean(),
                      "net mean": g["net_pips"].mean(), "net st.dev": g["net_pips"].std(), "win rate": (g["net_pips"] > 0).mean(),
                      "MFE >= 5": (g["best_pips"] >= 5).mean(), "MFE >= 10": (g["best_pips"] >= 10).mean(), "MFE >= 20": (g["best_pips"] >= 20).mean(),
                      "MAE <= -10": (g["worst_pips"] <= -10).mean(), "net / MFE": g["net_pips"].mean() / g["best_pips"].mean()})
S = T.groupby("signal").apply(summ).reindex(list(SIGNALS))
S.insert(0, "trade", [f"{'buy' if v[4] > 0 else 'sell'} {v[0]} {F.slot_label(v[1], v[2])} -> {F.slot_label(v[1], v[3]).split()[1]}" for v in SIGNALS.values()])
S.insert(1, "minutes", [v[3] - v[2] for v in SIGNALS.values()])
display(S.round(2)); S.to_csv(OUT / "11_summary.csv")
D = DR.groupby("pair")["day range"].describe(percentiles=[0.1, 0.5, 0.9])
print("whole New York day range (pips):"); display(D.round(1)); D.to_csv(OUT / "11_day_range.csv")

## 3. By year

Mean window range, MFE and net pips per year.

In [ ]:
for col in ("window range", "best_pips", "worst_pips", "net_pips"):
    y = T.pivot_table(index="signal", columns="year", values=col, aggfunc="mean").reindex(list(SIGNALS))
    print({"best_pips": "MFE", "worst_pips": "MAE", "net_pips": "net"}.get(col, col), "(pips, mean)"); display(y.round(2))
    y.to_csv(OUT / f"11_by_year_{col.replace(' ', '_')}.csv")
display(DR.pivot_table(index="pair", columns="year", values="day range", aggfunc="mean").round(1))

## 4. Distributions

For each signal: window range, MFE and the final net result per trade. The gap between MFE and net is how much of the open profit the fixed exit time gives back.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, sig in zip(axes.ravel(), SIGNALS):
    g = T[T["signal"] == sig]
    hi = g["window range"].quantile(0.98)
    bins = np.linspace(min(g["net_pips"].quantile(0.02), 0), hi, 60)
    ax.hist(g["window range"].clip(upper=hi), bins=bins, alpha=0.45, label=f"window range (mean {g['window range'].mean():.1f})")
    ax.hist(g["best_pips"].clip(upper=hi), bins=bins, alpha=0.45, label=f"MFE (mean {g['best_pips'].mean():.1f})")
    ax.hist(g["net_pips"].clip(lower=bins[0], upper=hi), bins=bins, alpha=0.45, label=f"net (mean {g['net_pips'].mean():.2f})")
    ax.axvline(0, color="black", lw=0.6); ax.set_title(sig); ax.legend(fontsize=7); ax.set_xlabel("pips")
plt.tight_layout(); plt.savefig(OUT / "11_distributions.png", dpi=110); plt.show()

In [ ]:
summary = {"study": "FXS11", "descriptive": True, "synthetic": SYNTHETIC, "signals": {k: list(v) for k, v in SIGNALS.items()},
           "summary": S.round(3).reset_index().to_dict(orient="records"), "day_range": D.round(2).reset_index().to_dict(orient="records")}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("written", OUT / "summary.json")